# 05 Deformation Story Map

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roymustang11/InSAR-Benchmark-Lab/blob/main/notebooks/05_deformation_story_map.ipynb)

This notebook assembles a final communication-oriented view: deformation map, validation time series, residuals, and benchmark summary.

**Important:** `DEMONSTRATION_DATA = True`. This is not a real deformation result. The figures are deterministic fixtures used to design the final reporting structure before real products are ingested.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/roymustang11/InSAR-Benchmark-Lab.git"
DEMONSTRATION_DATA = True

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    project_root = Path("/content/InSAR-Benchmark-Lab")
    if not project_root.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(project_root)])
else:
    cwd = Path.cwd().resolve()
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Project root: {project_root}")
print(f"Using demonstration fixture data: {DEMONSTRATION_DATA}")


## Demonstration Spatial And Validation Data

The story map needs four pieces: velocity grid, station locations, time-series comparison, and benchmark summary. Real data will replace these arrays after product extraction and GNSS alignment are implemented.


In [ ]:
import numpy as np
import pandas as pd

from insar_benchmark_lab.metrics import correlation, rmse, velocity_difference

lon = np.linspace(-121.0, -119.0, 80)
lat = np.linspace(35.0, 37.0, 70)
lon_grid, lat_grid = np.meshgrid(lon, lat)

bowl = np.exp(-(((lon_grid + 120.1) / 0.45) ** 2 + ((lat_grid - 36.0) / 0.55) ** 2))
regional_gradient = 0.15 * (lon_grid + 121.0) - 0.25 * (lat_grid - 35.0)
velocity_grid = -42.0 * bowl + regional_gradient

stations = pd.DataFrame(
    {
        "station": ["CV01", "CV02", "CV03"],
        "lon": [-120.15, -119.72, -120.55],
        "lat": [36.05, 35.72, 36.55],
        "gnss_velocity_mm_per_year": [-39.5, -24.0, -12.5],
        "insar_velocity_mm_per_year": [-41.0, -22.8, -14.2],
    }
)

dates = pd.date_range("2020-01-01", periods=10, freq="120D")
years = np.arange(len(dates)) * 120 / 365.25
story_gnss = -33.0 * years
story_insar = story_gnss + np.array([0.2, -0.5, 0.8, -0.7, 0.4, -0.2, 0.6, -0.4, 0.3, -0.6])


In [ ]:
benchmark_summary = {
    "study_area": "Central Valley demonstration fixture",
    "data_mode": "synthetic demonstration fixture, not a real deformation result",
    "station_count": int(len(stations)),
    "velocity_grid_min_mm_per_year": float(np.nanmin(velocity_grid)),
    "velocity_grid_max_mm_per_year": float(np.nanmax(velocity_grid)),
    "station_velocity_rmse_mm_per_year": rmse(stations["gnss_velocity_mm_per_year"], stations["insar_velocity_mm_per_year"]),
    "station_velocity_correlation": correlation(stations["gnss_velocity_mm_per_year"], stations["insar_velocity_mm_per_year"]),
    "example_time_series_velocity_difference_mm_per_year": velocity_difference(
        [item.date().isoformat() for item in dates], story_insar, story_gnss
    ),
}

pd.Series(benchmark_summary).to_frame("value")


## Story Figure

This layout is the final communication target for real results: map, station comparison, time series, and concise benchmark summary.


In [ ]:
import matplotlib.pyplot as plt

figure_panels, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)

map_ax = axes[0, 0]
im = map_ax.pcolormesh(lon_grid, lat_grid, velocity_grid, cmap="RdBu_r", shading="auto")
map_ax.scatter(stations["lon"], stations["lat"], c="black", s=45, label="GNSS validation sites")
for _, row in stations.iterrows():
    map_ax.text(row["lon"] + 0.03, row["lat"] + 0.03, row["station"], fontsize=8)
map_ax.set_title("LOS velocity fixture (mm/year)")
map_ax.set_xlabel("Longitude")
map_ax.set_ylabel("Latitude")
map_ax.legend(loc="lower left")
figure_panels.colorbar(im, ax=map_ax, label="mm/year")

scatter_ax = axes[0, 1]
scatter_ax.scatter(stations["gnss_velocity_mm_per_year"], stations["insar_velocity_mm_per_year"], s=60)
limits = [stations[["gnss_velocity_mm_per_year", "insar_velocity_mm_per_year"]].min().min() - 2,
          stations[["gnss_velocity_mm_per_year", "insar_velocity_mm_per_year"]].max().max() + 2]
scatter_ax.plot(limits, limits, color="black", linestyle="--")
scatter_ax.set_xlim(limits)
scatter_ax.set_ylim(limits)
scatter_ax.set_title("Station velocity comparison")
scatter_ax.set_xlabel("GNSS velocity (mm/year)")
scatter_ax.set_ylabel("InSAR velocity (mm/year)")
scatter_ax.grid(True, alpha=0.3)

ts_ax = axes[1, 0]
ts_ax.plot(dates, story_insar, marker="o", label="InSAR fixture")
ts_ax.plot(dates, story_gnss, marker="s", label="GNSS fixture")
ts_ax.set_title("Example displacement time series")
ts_ax.set_ylabel("LOS displacement (mm)")
ts_ax.grid(True, alpha=0.3)
ts_ax.legend()

summary_ax = axes[1, 1]
summary_ax.axis("off")
summary_lines = [
    "Benchmark summary",
    f"Mode: demonstration fixture",
    f"Stations: {benchmark_summary['station_count']}",
    f"Velocity RMSE: {benchmark_summary['station_velocity_rmse_mm_per_year']:.2f} mm/year",
    f"Velocity correlation: {benchmark_summary['station_velocity_correlation']:.2f}",
    f"Example velocity diff: {benchmark_summary['example_time_series_velocity_difference_mm_per_year']:.2f} mm/year",
    "Real result status: pending OPERA/GNSS extraction",
]
summary_ax.text(0.02, 0.95, "\n".join(summary_lines), va="top", fontsize=12)

figure_panels.suptitle("Demonstration Deformation Story Map Layout")
plt.show()


## Transition To Real Story Map

A real story map should keep this structure but replace the fixture arrays with OPERA/MintPy velocities, GNSS station velocities, and time-series validation outputs. The final figure should state product version, access date, reference-area choice, masking threshold, and validation metrics.
